# Laboratorio: la señal de temperatura

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jfbonder/imc/blob/main/notebooks/lab-senal-temperatura.ipynb)

Este laboratorio cierra el problema conductor de la Parte II: **la temperatura del aire, medida cada hora durante tres años en Aeroparque**. En el texto vimos las herramientas (series de Fourier, transformada de Fourier, DFT, filtrado y compresión) y los notebooks `06-series`, `07-transformada-DFT` y `08-aplicaciones` las aplican a esta serie y muestran los resultados principales. Acá **las usás vos**, con más cuidado, sobre datos reales, y aparecen todas las cosas que con una señal sintética no aparecen: una media que no es cero, una tendencia (¿o no?), picos que no caen exactamente en la grilla, ruido que no es blanco, fuga espectral.

Las cinco preguntas del problema conductor (al comienzo de la Parte II de las notas) son el hilo del laboratorio, y la parte que se evalúa es una respuesta escrita a cada una:

1. ¿Qué frecuencias contiene la serie? ¿Con qué amplitud y fase contribuye cada una? ¿Cuánto de la variabilidad total explican el ciclo diario y el anual?
2. ¿Cómo separar la parte "regular" (los ciclos) de la parte irregular, y qué queda cuando se lo hace?
3. Si en lugar de una medición por hora tuviéramos una por día, o una por semana, ¿qué información se pierde y qué información *se falsea*?
4. ¿Cuántos números hacen falta para reconstruir la señal con un error dado? ¿Se puede comprimir?
5. Las herramientas que sirven para esta señal, ¿sirven igual para un audio, un electrocardiograma o una imagen?

**Qué vamos a hacer.** (1) Cargar los datos, mirarlos y *elegir el tramo* sobre el que se calcula la DFT; (2) calcular el espectro de potencia, en ciclos por día y en escala logarítmica, e identificar los picos y la forma del piso; (3) medir amplitud y fase de los ciclos anual y diario, de dos maneras (con la DFT y por mínimos cuadrados) y ver cómo cambia el ciclo diario con la estación; (4) calcular la varianza explicada con la identidad de Parseval; (5) entender la fuga espectral y qué hace una ventana de Hann; (6) submuestrear la serie y ver el aliasing en datos reales; (7) filtrar en frecuencia, estudiar el residuo y compararlo con los promedios móviles (que también son filtros: ¿cuál es su respuesta en frecuencia?); (8) comprimir por coeficientes de mayor módulo; (9) hacer el espectrograma de la temperatura y de una cuerda de guitarra, y discutir el compromiso tiempo–frecuencia.

**Lo que las notas no explican y este notebook sí.** Cómo pasar de la DFT a unidades físicas (ciclos por día, °C, horas del día, fechas); cómo elegir el tramo de datos y por qué importa que sea múltiplo de los períodos; qué es el periodograma y por qué se mira en escala logarítmica; qué es la fuga espectral y las ventanas; cómo se estima una amplitud y una fase con la DFT y por mínimos cuadrados con frecuencia fija (y cuándo dan distinto); la respuesta en frecuencia de un promedio móvil; y el espectrograma (STFT). El texto de cada sección es la única presentación que van a tener de esos temas: leanlo, no solo las consignas.

**Herramientas disponibles.** `imc.espectro` (`dft`, `amplitudes`, `periodograma`, `welch`, `filtrar`, `espectrograma`; leé los docstrings: las convenciones de normalización están ahí), `imc.datos.obtener` (los datos), `imc.estilo` (colores y figuras), `numpy.fft`, `scipy.io.wavfile` (audio) y `pandas` (fechas). No vamos a reconstruir lo que ya hacen los notebooks del texto (`07-transformada-DFT`: espectro de amplitudes y aliasing con la serie de 2023; `08-aplicaciones`: filtro de picos, residuo, compresión de la serie y de la imagen): allí están los resultados de referencia, y conviene tenerlos a mano para comparar. La diferencia es que acá se trata de que los produzcas vos, entiendas por qué dan lo que dan y los interpretes.

**Cómo se evalúa.** Como siempre: la sección final de **interpretación escrita**. Cada tarea dice qué se espera y trae una celda de verificación; si un `assert` falla, el mensaje te dice qué revisar. Las celdas de verificación no reemplazan la interpretación: un número correcto sin explicación no alcanza. Tiempo estimado: **dos sesiones de 3 a 4 h**: las Tareas 1 a 6 (la primera mitad: espectro, amplitud y fase, varianza, ventanas, aliasing) y las Tareas 7 a 9 (filtrado, compresión, espectrograma).

In [ ]:
# Configuración (funciona en Colab y en una copia local del repositorio)
try:
    import imc
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/jfbonder/imc.git"], check=True)
    import imc

import numpy as np
import matplotlib.pyplot as plt
from imc import estilo, datos
from imc.estilo import COLORES
import pandas as pd
import matplotlib.dates as mdates
from scipy.io import wavfile
from imc import espectro
from imc.estilo import CICLO
estilo.activar()

## 1. Los datos, las unidades y la elección del tramo

### Los datos

`temperatura_horaria.csv` (ver `datos/README.md`) tiene la temperatura del aire a 2 m, en °C, cada hora, del 1/1/2021 al 31/12/2023: $3 \times 365 \times 24 = 26\,280$ valores, sin huecos. Ojo con dos cosas. Son datos de **reanálisis** (un modelo numérico que asimila observaciones, ERA5), así que son más suaves y regulares que los de un termómetro real: sirven perfectamente para lo que hacemos, pero no los tomes por una medición directa. Y la hora es **local sin cambio de horario** (en Argentina no lo hay desde 2009), de modo que "a las 15:00" significa lo mismo en enero que en julio y el ciclo diario no se corre artificialmente.

### Unidades: de la DFT a ciclos por día

La DFT de $N$ muestras $x_j = x(t_j)$, $t_j = j\,\Delta t$, es $\hat x[k] = \sum_{j=0}^{N-1} x_j\, e^{-2\pi i jk/N}$. El coeficiente $k$ corresponde a la frecuencia

$$\xi_k = \frac{k}{T} = \frac{k}{N\,\Delta t}, \qquad T = N\,\Delta t \ \text{(duración del tramo)},$$

en las unidades de $1/\Delta t$. **Nosotros medimos el tiempo en días**: $\Delta t = 1/24$ día, y $\xi$ queda en **ciclos por día**. Si $T$ son 365 días, $\xi_k = k/365$ ciclos por día $= k$ ciclos por año: el ciclo anual es $k = 1$; el diario, $\xi = 1$ ciclo/día, es $k = 365$; el de 12 horas, $k = 730$. Tres cosas para tener siempre presentes:

* la **resolución en frecuencia** es $\Delta\xi = 1/T$: dos frecuencias más cercanas que eso no se distinguen;
* la frecuencia más alta representable es la de **Nyquist**, $\xi_{\max} = 1/(2\Delta t) = 12$ ciclos/día (período de 2 horas);
* para una señal real, $\hat x[N-k] = \overline{\hat x[k]}$: alcanza con la mitad $k = 0, \dots, N/2$ (`np.fft.rfft`, que es lo que usa `imc.espectro`).

### Media, tendencia y elección del tramo

**La media.** El coeficiente $\hat x[0] = \sum_j x_j = N\bar x$ es la media (por $N$). Con una temperatura de unos 18 °C y fluctuaciones de unos 6 °C, la media aporta el 90 % de $\sum x_j^2$: si no se la resta, el espectro queda dominado por un pico en $k = 0$ que no dice nada sobre las oscilaciones. `espectro.periodograma` la resta por defecto.

**La tendencia.** Una serie que sube o baja lentamente no es periódica, y la DFT (que supone que la señal se repite con período $T$) ve un *salto* entre el último y el primer valor. Ese salto reparte energía en todas las frecuencias (la fuga de la Sección 5). En series largas se resta una tendencia lineal antes de la DFT; en la nuestra hay que preguntarse antes si hay tendencia: con tres años, ¿una pendiente de unas décimas de grado por año es una tendencia climática o la variación de un año a otro? (Mirá cuánto varía la media anual.)

**El tramo.** La DFT trabaja con *todo* el vector que se le da. Si $T$ es un múltiplo de un período, la frecuencia correspondiente cae exactamente en un $k$ entero y aparece en un solo coeficiente; si no, se reparte entre coeficientes vecinos (fuga). Hay dos elecciones naturales:

* **Un año exacto (2023):** $N = 8760$, $T = 365$ días. El ciclo anual es $k = 1$ y el diario $k = 365$, ambos exactamente en la grilla. La resolución es $1/365$ ciclos/día.
* **Los tres años:** $N = 26\,280$, $T = 1095$ días. La resolución mejora a $1/1095$ (el anual es $k = 3$, el diario $k = 1095$), pero el ciclo anual **no es el mismo** cada año (el 2022 fue más frío que el 2023) y ese ciclo "casi periódico" se ensancha en un pico más ancho y más bajo.

Trabajamos con 2023, que es la elección de las notas y la de los notebooks del texto. Al final de la tarea van a poder decir por qué.

### Tarea 1: Cargar, graficar y elegir el tramo

La celda ya carga el archivo con `pandas` (columnas `fecha_hora`, `temp`). Completá:

1. Una figura de dos paneles: arriba la serie completa (2021–2023), abajo un zoom de dos semanas (por ejemplo del 1 al 14 de enero de 2023). Ejes con nombre y unidades.
2. La media y la varianza (`T.var()`, sin corrección: la de la identidad de Parseval) de cada año y de toda la serie, y la pendiente de la recta de mínimos cuadrados de toda la serie en °C por año (`np.polyfit` contra `anios`, el tiempo en años). ¿Es una tendencia?
3. Las variables del tramo de 2023 que vamos a usar en todo el resto: `T` (los 8760 valores, `float`), `fecha` (las fechas correspondientes, una `Series` de `pandas`), `N`, `dt` (en días) y `t` (tiempos en días desde el 1/1/2023 a las 00:00).

**Qué se espera.** Una serie con un ciclo anual bien claro (más cálida en el verano austral, es decir, en enero–febrero) y, en el zoom, un ciclo diario con máximos a media tarde, montado sobre variaciones irregulares de varios días (frentes). Medias anuales del orden de 17–18 °C, varianzas de 36–40 °C² y una pendiente de unas décimas de grado por año, del mismo tamaño que la diferencia entre años: no hay evidencia de tendencia con tres años. Anotá en una línea por qué elegís 2023 y no los tres años.

In [ ]:
ruta = datos.obtener("temperatura_horaria.csv")
df = pd.read_csv(ruta, parse_dates=["fecha_hora"])
print(df.shape, df.fecha_hora.iloc[0], "->", df.fecha_hora.iloc[-1])

# TODO: media y varianza de cada año (df.groupby(df.fecha_hora.dt.year)) y de toda la serie
# TODO: pendiente = ... (np.polyfit de df.temp contra anios, en °C por año)
anios = np.arange(len(df)) / 24 / 365

# TODO: el tramo de 2023
# T = ...; fecha = ...; N = ...; dt = ...; t = ...

# TODO: figura de dos paneles (serie completa y zoom de dos semanas)

In [ ]:
# Verificación
assert N == 8760 and abs(dt - 1 / 24) < 1e-12 and t.shape == (N,), "T, N, dt, t: el tramo debe ser 2023 (8760 horas, dt en días)"
assert fecha.iloc[0] == pd.Timestamp("2023-01-01 00:00") and fecha.iloc[-1] == pd.Timestamp("2023-12-31 23:00"), "fecha mal cortada"
assert abs(T.mean() - 18.24) < 0.01 and abs(T.var() - 39.95) < 0.05, "media y varianza de 2023: ~18.24 °C y ~39.95 °C²"
assert abs(pendiente) < 0.5, "la pendiente debe ser de décimas de °C por año (¿la calculaste contra el tiempo en años?)"
print(f"media 2023 = {T.mean():.2f} °C, varianza = {T.var():.2f} °C² (desvío {T.std():.2f} °C), pendiente {pendiente:+.2f} °C/año: OK")

## 2. El espectro de potencia (periodograma)

### Qué se grafica

Para $x$ real de $N$ muestras con la media restada, el **espectro de potencia** (o **periodograma**) es

$$P[k] = \frac{|\hat x[k]|^2}{N}, \qquad \xi_k = \frac{k}{N\,\Delta t}.$$

Es el que calcula `espectro.periodograma(x, dt)` (con la normalización que hace que valga la identidad de Plancherel–Parseval discreta $\sum_{k=0}^{N-1} P[k] = \sum_j x_j^2$; en la mitad $0 < k < N/2$ que se devuelve cada valor cuenta dos veces, por $\pm k$). $P[k]$ mide *cuánta energía de la señal hay en la frecuencia $\xi_k$*; un coseno de amplitud $A$ y frecuencia $\xi_k$ exactamente en la grilla produce $P[k] = A^2 N/4$: la altura de un pico depende de $N$ y de $A^2$, no es una "densidad" que se pueda comparar entre tramos de distinta longitud sin dividir por algo (Sección 5).

### Por qué escala logarítmica

Los picos del ciclo anual y del diario son cuatro o cinco órdenes de magnitud más altos que el "piso" de las fluctuaciones irregulares. En escala lineal solo se ve un par de palitos y el resto es una línea sobre el cero; con `semilogy` (potencia en log) se ven a la vez los picos, sus armónicos y la forma del piso, y con `loglog` (frecuencia también en log) una **ley de potencias** $P \propto \xi^{-\alpha}$ se ve como una recta de pendiente $-\alpha$. Es la forma estándar de mirar espectros de señales naturales.

### Qué esperar

* **Picos** en el ciclo anual ($\xi = 1/365$), en el diario ($\xi = 1$) y en sus armónicos: los múltiplos enteros de $1/365$ (ciclos del año que no son sinusoidales) y de $1$ ciclo/día (el ciclo diario no es un coseno puro: sube más despacio de lo que baja, y eso genera armónicos de 12 h, 8 h, 6 h, ...). Un armónico *existe* si sobresale del piso vecino; para decidirlo, compará $P[k]$ con la mediana de $P$ en los $\pm 20$ coeficientes que lo rodean (el "realce"). Ojo: no todo máximo local del periodograma es un pico genuino. El periodograma de una señal aleatoria es muy irregular (cada $P[k]$ fluctúa alrededor de su valor medio con una dispersión del orden del propio valor) y por puro azar aparecen máximos por todos lados, sobre todo donde el piso es alto, a bajas frecuencias. Un pico *real* sobresale del piso vecino por un factor grande, no por un factor 2 o 3.
* **Piso** que baja con la frecuencia: las fluctuaciones lentas (semanas) tienen mucha más energía que las rápidas (horas). Un espectro que decae como una ley de potencias se llama, por analogía con la luz, "ruido rojo"; uno plano, "ruido blanco". ¿De qué tipo es el de la temperatura? Se mide ajustando una recta a $\log P$ contra $\log \xi$ en un rango sin picos.

### Tarea 2: Espectro de potencia de 2023

1. Calculá `xi, P = espectro.periodograma(T, dt)` (`xi` sale en **ciclos por día** porque `dt` está en días). Verificá la resolución y la frecuencia de Nyquist.
2. Graficá $P$ contra $\xi$ en dos paneles: `semilogy` con $\xi \in [0, 12]$ y `loglog` (sin $\xi = 0$). Marcá los picos que identifiques con puntos y anotalos.
3. Escribí la función `realce(k, m=20)` que devuelve $P[k]$ dividido por la mediana de $P$ en los $m$ coeficientes a cada lado de $k$ (sin contar $k$) y evaluala para los candidatos $k = 1, 2, 3$ (anual, semianual y de cuatro meses) y $k = 365 j$, $j = 1, \dots, 5$ (diario y sus armónicos). ¿Cuáles sobresalen? ¿Es el semianual un pico o una fluctuación?
4. ¿Aparece algún pico que no esperabas? Listá los cinco valores más altos de $P$ con $\xi < 0.5$ que no sean los candidatos (excluí $\pm 3$ coeficientes alrededor de cada candidato): ¿a qué períodos corresponden y cuánto sobresalen (realce)? ¿Son ciclos o fluctuaciones?
5. Ajustá la pendiente $-\alpha$ del piso en escala log-log en dos rangos sin picos: $\xi \in [0.05, 0.8]$ y $\xi \in [5, 11.5]$ ciclos/día (`np.polyfit(np.log(xi[m]), np.log(P[m]), 1)`), y guardalas en `alfa_bajo` y `alfa_alto` (positivas).

**Qué se espera.** Picos gigantescos en $\xi = 1/365$ y $\xi = 1$ (realce de cientos); realce claro en los armónicos de 12 h y de 6 h y más modesto en el de 8 h; **el semianual y el de cuatro meses casi no sobresalen** de las fluctuaciones vecinas (realce de 2 y menor que 1), por más que en la tabla de amplitudes el semianual valga $0.8$ °C: con un solo año no se distingue de la variabilidad de baja frecuencia. Los otros máximos que encuentres a baja frecuencia (períodos de 2 a 6 semanas) tienen realces de 3 a 10 y son las fluctuaciones del piso, no ciclos. Y un piso con pendiente cercana a $-2$ ($\alpha \approx 1.8$–$2$ en los dos rangos). Respondé con una tabla (k, $\xi$, período, $P[k]$, realce) y una línea sobre la forma del piso: "no es blanco porque...".

In [ ]:
# TODO: xi, P = espectro.periodograma(T, dt)
# TODO: figura de dos paneles (semilogy y loglog), con los picos marcados

def realce(k, m=20):
    # P[k] / mediana de P en los m coeficientes a cada lado de k
    ...  # TODO

candidatos = [(1, "anual"), (2, "semianual"), (3, "4 meses"), (365, "diario"), (730, "12 h"),
              (1095, "8 h"), (1460, "6 h"), (1825, "4.8 h")]
# TODO: imprimir una tabla k, xi, período (en días u horas), P[k], realce

# TODO: los cinco valores más altos de P con xi < 0.5 fuera de los candidatos (excluir ±3 coeficientes)
# TODO: alfa_bajo, alfa_alto (pendientes del piso, en positivo)

In [ ]:
# Verificación
assert xi.shape == P.shape == (N // 2 + 1,) and abs(xi[1] - 1 / 365) < 1e-9 and abs(xi[-1] - 12) < 1e-9, "xi debe estar en ciclos por día (dt en días)"
assert realce(365) > 50 and realce(1) > 5, "los picos diario y anual deben sobresalir muchísimo del piso"
assert realce(730) > 50 and realce(1095) > 5 and realce(1460) > 10, "los armónicos de 12 h, 8 h y 6 h sobresalen del piso"
assert 1.2 < alfa_bajo < 2.6 and 1.2 < alfa_alto < 2.6, "el piso decae con una pendiente cercana a -2: ¿pasaste logaritmos de xi y de P?"
print("espectro de potencia: OK")

## 3. Amplitud y fase de una componente

Si en la señal hay una componente $A\cos(2\pi\xi t + \varphi)$ con $\xi = \xi_k$ en la grilla, la DFT la ve en el coeficiente $k$, y con la convención de `numpy` ($\hat x[k] = \sum_j x_j e^{-2\pi i jk/N}$, $t_j = j\Delta t$ con el origen en el primer dato)

$$A = \frac{2\,|\hat x[k]|}{N}, \qquad \varphi = \arg \hat x[k].$$

(El factor 2 aparece porque la energía de un coseno se reparte entre $+k$ y $-k$.) Es lo que devuelve `espectro.amplitudes(x, dt)`. La componente alcanza su **máximo** cuando el argumento del coseno vale $0$ módulo $2\pi$: $2\pi\xi t^* + \varphi = 0$, es decir

$$t^* = -\frac{\varphi}{2\pi\xi} \pmod{1/\xi}.$$

Con $t = 0$ en el 1/1/2023 a las 00:00, $t^*$ está en días desde esa fecha. Para el ciclo anual, $t^*$ da la fecha; para el diario ($1/\xi$ = 1 día), la parte fraccionaria de $t^*$ por 24 es la hora del día. Cuidado con el signo de $\varphi$ y con reducir módulo el período: es el error más común.

### Estimar por mínimos cuadrados con frecuencia fija

Otra manera de estimar la misma componente, que conocen de Estadística, es un **modelo lineal**: como $A\cos(2\pi\xi t + \varphi) = a\cos(2\pi\xi t) + b\sin(2\pi\xi t)$ con $a = A\cos\varphi$, $b = -A\sin\varphi$, si $\xi$ es conocida los parámetros $(a, b)$ (y la media $c$) resultan de una regresión lineal $x_j \approx c + a\cos(2\pi\xi t_j) + b\sin(2\pi\xi t_j)$, con $A = \sqrt{a^2 + b^2}$ y $\varphi = \operatorname{atan2}(-b, a)$. Se pueden agregar varias frecuencias como columnas de la matriz de diseño. Si las frecuencias están en la grilla del tramo, las columnas son ortogonales entre sí y **DFT y mínimos cuadrados dan exactamente lo mismo**; si no, dan distinto, y el de mínimos cuadrados tiene razón (siempre que se conozca la frecuencia):

* **Con la DFT**, una frecuencia que no está en la grilla se reparte entre coeficientes vecinos y el pico más alto subestima la amplitud (hasta un 36 % si la frecuencia queda justo entre dos coeficientes).
* **Por mínimos cuadrados** se ajusta la frecuencia exacta, sin depender de la grilla; a cambio, hay que *saber* qué frecuencias poner.

Esto importa en la tercera consigna: para ver cómo cambia el ciclo diario en la estación fría y en la cálida hay que mirar tramos cortos, y ahí la grilla es gruesa ($\Delta\xi = 1/T$).

### Tarea 3: Amplitud y fase de los ciclos anual y diario

1. Con `xi_a, A, phi = espectro.amplitudes(T, dt)`, calculá la amplitud $A_1$ y la fase del ciclo anual ($k = 1$) y del diario ($k = 365$). A partir de la fase, obtené el **día del año del máximo anual** (`fecha_max_anual`, un `pd.Timestamp`: `fecha.iloc[0] + pd.Timedelta(days=...)`) y la **hora del máximo diario** (`hora_max_diaria`, en horas decimales). Guardá también `A_anual`, `A_diaria`.
2. Escribí `ajuste_ls(x, t, frecs)` que ajusta por mínimos cuadrados (`np.linalg.lstsq`) $x \approx c + \sum_f a_f \cos(2\pi f t) + b_f\sin(2\pi f t)$ y devuelve `(c, A, phi)` con `A`, `phi` arreglos (uno por frecuencia). Aplicala a `frecs = [1/365, 1]` (guardá `A_ls`, `phi_ls`) y verificá que coincide con la DFT hasta el error de redondeo. ¿Por qué coincide?
3. **Estacionalidad del ciclo diario.** Recortá enero y julio de 2023 (`fecha.dt.month`), y calculá para cada mes la amplitud del ciclo diario con la DFT del mes (¿en qué $k$ cae $\xi = 1$ ciclo/día en un tramo de 31 días?). Guardá `A_enero`, `A_julio`. Calculá también la hora del máximo diario de cada mes.
4. **Cuando la grilla no alcanza.** Tomá un tramo de 10 días y medio (252 horas) de enero, `T[120:120 + 252]`: ahí $\xi = 1$ ciclo/día queda justo entre dos coeficientes. Compará la amplitud diaria que da el pico más alto de la DFT (`espectro.amplitudes`, coeficiente más cercano a $\xi = 1$) con la del ajuste `ajuste_ls` con $\xi = 1$ fijo (`A_dft_corto`, `A_ls_corto`) y con `A_enero`.

**Qué se espera.** $A_1 \approx 6.9$ °C con máximo el 25 de enero; $A_{365} \approx 2.8$ °C con máximo alrededor de las 15:45; el ciclo diario es más marcado en enero ($\approx 3.0$ °C) que en julio ($\approx 2.3$ °C). En el tramo de 10.5 días, la DFT subestima mucho la amplitud (alrededor de $2$ °C) y mínimos cuadrados da un valor cercano al de enero completo (más de $3$ °C).

In [ ]:
xi_a, A, phi = espectro.amplitudes(T, dt)
# TODO: A_anual, A_diaria, fecha_max_anual (pd.Timestamp), hora_max_diaria (horas decimales)

def ajuste_ls(x, t, frecs):
    # x ~ c + sum_f a_f cos(2 pi f t) + b_f sin(2 pi f t); devuelve (c, A, phi)
    ...  # TODO

# TODO: comparar ajuste_ls(T, t, [1/365, 1]) con la DFT

# TODO: A_enero, A_julio (DFT de cada mes) y hora del máximo diario de cada mes

# TODO: tramo corto de 252 horas: A_dft_corto, A_ls_corto

In [ ]:
# Verificación
assert abs(A_anual - 6.92) < 0.05 and abs(A_diaria - 2.76) < 0.05, "amplitudes: A_1 ~ 6.9 °C, A_365 ~ 2.8 °C"
assert fecha_max_anual.month == 1 and 20 <= fecha_max_anual.day <= 30, "el máximo anual cae a fines de enero (cuidado con el signo de la fase y con el módulo)"
assert 15 <= hora_max_diaria <= 16.5, "el máximo diario cae a media tarde (~15:45)"
assert np.allclose(A_ls, [A[1], A[365]], atol=1e-6) and np.allclose(phi_ls, [phi[1], phi[365]], atol=1e-6), "DFT y mínimos cuadrados deberían coincidir sobre la grilla"
assert A_enero > A_julio + 0.4, "el ciclo diario es más marcado en enero que en julio"
assert A_dft_corto < 0.8 * A_ls_corto and abs(A_ls_corto - A_enero) < 0.5, "en el tramo corto la DFT subestima; mínimos cuadrados no"
print("amplitud y fase: OK")

## 4. Varianza explicada e identidad de Parseval

La **identidad de Plancherel discreta** dice que $\sum_{j} x_j^2 = \frac1N\sum_{k=0}^{N-1}|\hat x[k]|^2$. Con la media restada ($\hat x[0] = 0$), el miembro izquierdo es $N\operatorname{Var}(x)$ (con $\operatorname{Var}$ **sin** corrección $N/(N-1)$: `np.var` por defecto), y queda

$$\operatorname{Var}(x) = \frac1{N^2}\sum_{k\ne0}|\hat x[k]|^2 .$$

Cada frecuencia contribuye a la varianza con su parte: para una señal real, $k$ y $N-k$ contribuyen igual, así que un ciclo en $\xi_k$ (con $0<k<N/2$) explica la fracción $2P[k]/\sum_{k\neq0}P[k]$ de la varianza. Fijate que **es un porcentaje de la varianza (la variabilidad alrededor de la media), no de la señal**, y que coincide con el $R^2$ de la regresión con esas componentes: para una componente de amplitud $A$ sola, $\operatorname{Var} = A^2/2$ (el promedio de $\cos^2$ es $1/2$). Es una manera de chequear la cuenta: con $A_1$ y $A_{365}$ de la tarea anterior tiene que dar lo mismo.

Un par de cuidados: (i) la varianza explicada por *sumar* componentes es la suma de las de cada una porque las columnas de la DFT son ortogonales; (ii) los picos de la DFT tienen "hombros" (frecuencias vecinas) que no se cuentan si se toma un solo $k$: eso subestima el porcentaje real de los ciclos.

### Tarea 4: Varianza explicada

1. Verificá numéricamente Parseval con la serie de 2023: `X = np.fft.fft(T)` y que $\sum_{k\ne0}|\hat T[k]|^2/N$ coincide con `N * T.var()` (`E_tot`).
2. Escribí `fraccion(ks)` que devuelve la fracción de la varianza explicada por las componentes en los índices `ks` (contando $\pm k$), y calculá la de cada ciclo por separado (anual $k=1$; semianual $2$; diario $365$; 12 h $730$; 8 h $1095$) y las acumuladas: anual + diario; anual + diario + armónicos de 12 h y 8 h; y todos los anteriores más el semianual. Guardá esta última en `frac_picos`.
3. Verificá con la fórmula $A^2/2$ (dividida por la varianza) el porcentaje del ciclo anual y del diario.
4. Hacé un gráfico de barras (o una tabla) con el aporte de cada uno y el resto ("irregular").

**Qué se espera.** El ciclo anual explica cerca del 60 % y el diario cerca del 9.5 %; anual + diario, 69–70 %; con todos los picos, entre el 70 y el 71 %. Es decir, casi un tercio de la varianza de la temperatura horaria **no** es regular: es la variabilidad de los frentes, que en el espectro no forma picos. Anotá el porcentaje y una frase sobre qué significa que un ciclo "explique el 60 %".

In [ ]:
X = np.fft.fft(T)
# TODO: E_tot = sum_{k != 0} |X[k]|^2 / N  y comprobar que es N * T.var()

def fraccion(ks):
    # fracción de la varianza explicada por los índices ks (contando +k y -k)
    ...  # TODO

# TODO: fracciones por separado y acumuladas; frac_picos (todos los picos: k = 1, 2, 365, 730, 1095)
# TODO: chequeo con A^2 / 2 / varianza
# TODO: gráfico de barras

In [ ]:
# Verificación
assert np.isclose(E_tot, N * T.var()), "Parseval: la suma sobre k != 0 de |X|^2/N es N * var (con ddof = 0)"
assert 0.59 < fraccion([1]) < 0.61 and 0.09 < fraccion([365]) < 0.10, "anual ~60 %, diario ~9.5 %"
assert np.isclose(fraccion([1]), A_anual ** 2 / 2 / T.var(), atol=1e-6), "el % del anual debe coincidir con A^2/2/var"
assert 0.70 < frac_picos < 0.72, "todos los picos: ~71 %"
print(f"varianza explicada por los picos: {100 * frac_picos:.1f} %: OK")

## 5. Fuga espectral y ventana de Hann

### Por qué aparece la fuga

La DFT de un tramo de $N$ muestras equivale a multiplicar la señal (infinita) por una **ventana rectangular** que vale 1 en el tramo y 0 fuera, y transformar. El producto en el tiempo es una convolución en frecuencia: cada frecuencia de la señal aparece en el espectro **ensanchada por la transformada de la ventana**, que para la rectangular es un $\operatorname{sinc}$: un lóbulo central de ancho $2/T$ y lóbulos laterales que decaen lentamente (como $1/(\xi - \xi_0)$ en amplitud). Si $\xi_0$ es un múltiplo de $1/T$, la grilla de la DFT cae justo en los ceros del $\operatorname{sinc}$ y solo se ve el centro: un pico limpio. Si no, la grilla muestrea el sinc en puntos intermedios y **la energía de la componente "se fuga" a los coeficientes vecinos** en una cola que decae lentamente. Efectos en la práctica:

* el pico más alto **subestima** la amplitud (hasta 36 % si $\xi_0$ queda justo entre dos coeficientes);
* la cola de un pico grande **tapa** picos pequeños vecinos y eleva el piso: en un espectro con picos de $10^5$ y un piso de $10^{-1}$, una cola que decae como $1/\xi$ en amplitud ($1/\xi^2$ en potencia) puede dominar el piso lejos del pico;
* es equivalente a mirar el "salto" entre el final y el comienzo del tramo: una señal no periódica en $T$ tiene un salto, y un salto tiene un espectro que decae como $1/\xi$ en amplitud.

### Ventanas

El remedio es **suavizar los bordes**: multiplicar el tramo por una ventana $w[j]$ que baja a cero suavemente en los dos extremos. La **ventana de Hann** es $w[j] = \frac12\bigl(1 - \cos\frac{2\pi j}{N-1}\bigr)$ (`np.hanning(N)`). Su transformada tiene lóbulos laterales que decaen mucho más rápido (como $1/\xi^3$ en amplitud), a cambio de un **lóbulo central más ancho** (el doble: $4/T$ en lugar de $2/T$: pierde resolución). Es el compromiso de siempre: menos fuga contra menos resolución. Al multiplicar por $w$ el pico también baja (la ventana atenúa la señal, en promedio a la mitad de su amplitud), y hay que normalizar para comparar: `espectro.periodograma(x, dt, ventana="hann")` divide por $N\,\overline{w^2}$ para que la potencia total se conserve; si en cambio querés **la amplitud** de una componente de un espectro con ventana, la estimación es $A \approx 2\,|\hat{(xw)}[k]|/\sum_j w[j]$.

Una ventaja menos conocida de Hann: el error por "estar entre dos coeficientes" es mucho menor (la subestimación máxima de la amplitud pasa de 36 % a 15 %).

### Cómo medirlo

Antes de mirar la temperatura conviene un **control con una señal cuyo espectro se conoce**: un coseno puro de amplitud 1 y frecuencia $\xi_0 = 1$ ciclo/día en un tramo que *no* contiene un número entero de períodos (10 días y medio), con y sin ventana. Todo lo que aparezca fuera de $\xi_0$ es fuga. Después se mira lo mismo con la serie real.

### Tarea 5: Rectangular contra Hann

1. **Control sintético.** Construí `x = np.cos(2*np.pi*t_c)` con `t_c = np.arange(252) * dt` (10.5 días, $\xi_0 = 1$, amplitud 1). Calculá los periodogramas rectangular y con Hann (`espectro.periodograma(..., ventana="hann")`), y graficalos juntos con `semilogy` para $\xi \in [0, 4]$. Leé la amplitud del pico más alto en cada caso: para el rectangular $A = 2|\hat x[k]|/N$; para Hann, $A = 2|\widehat{xw}[k]|/\sum w$ (las variables `A_rect_sint`, `A_hann_sint`). Repetí con un tramo de 240 horas (10 días exactos) como referencia.
2. **Serie real, un año.** Los periodogramas de `T` (2023) rectangular y con Hann, graficados juntos cerca del pico diario ($\xi \in [0.9, 1.1]$) y cerca del pico anual ($\xi \in [0, 0.02]$). Anotá: alturas de los picos, cuántos coeficientes forman cada pico, y cuánta potencia hay en las bandas "piso" $\xi \in [3.3, 3.6]$ y $[8, 9]$ con cada ventana (`piso_rect`, `piso_hann`, en la banda $[3.3, 3.6]$).
3. **Serie real, un tramo que no es un año exacto.** Repetí (2) con `T[1416:1416 + 2412]` (100.5 días desde el 1/3/2023): el ciclo diario cae otra vez entre dos coeficientes. ¿Qué cambia con la ventana en el pico y lejos del pico?

**Qué se espera.** En el control: con el tramo de 10.5 días, el rectangular lee una amplitud cercana a $0.64$ y una cola que decae lentamente ("cola de sinc"), y Hann lee una amplitud cercana a $0.85$ con una cola que cae varios órdenes de magnitud más rápido, aunque con un pico más ancho; con 10 días exactos, los dos leen $\approx 1$ (Hann reparte el pico en tres coeficientes). En 2023 (con los picos en la grilla) el rectangular ve un pico de un solo coeficiente y Hann uno de tres (más ancho y más bajo, con las "alas" que suben), y el piso lejos de los picos baja poco (del orden del 10 al 30 %): en la serie real el piso ya es mucho más alto que la fuga del sinc, y solo en el control se ve limpia. Lo que importa es *dónde* ayuda la ventana: cerca de picos fuertes que no están en la grilla.

In [ ]:
n_c = 252                              # 10.5 días
t_c = np.arange(n_c) * dt
x_sint = np.cos(2 * np.pi * t_c)       # coseno de amplitud 1 y frecuencia 1 ciclo/día

# TODO: periodogramas rectangular y hann del control; A_rect_sint, A_hann_sint (amplitud leída en el pico más alto)
#       A = 2|X[k]|/N para el rectangular;  A = 2|X_w[k]|/sum(w) para Hann, con X_w = rfft(x w)
# TODO: lo mismo para 240 horas (10 días exactos)
# TODO: figura del control (semilogy, xi en [0, 4])

# TODO: serie real 2023: rectangular vs hann cerca de xi = 1 y de xi = 1/365; piso_rect, piso_hann en [3.3, 3.6]
# TODO: tramo T[1416:1416+2412]

In [ ]:
# Verificación
assert 0.55 < A_rect_sint < 0.75 and 0.8 < A_hann_sint < 0.92, "control (10.5 días): rectangular ~0.64 y Hann ~0.85 (con A = 2|X|/sum(w))"
assert piso_hann < piso_rect, "el piso lejos de los picos baja con Hann"
print("fuga espectral y ventanas: OK")

## 6. Submuestreo y aliasing en datos reales

Si una señal se muestrea cada $\Delta t$, las frecuencias solo pueden distinguirse hasta la de Nyquist $\xi_N = 1/(2\Delta t)$, y una componente de frecuencia $\xi$ **fuera** de $[0, \xi_N]$ no desaparece: se **pliega** sobre otra dentro de la banda. Con frecuencia de muestreo $f_s = 1/\Delta t$, la frecuencia observada (el *alias*) es

$$\xi_{\text{alias}} = \bigl|\,\xi - f_s\, \operatorname{round}(\xi/f_s)\bigr| \in [0, \xi_N].$$

Por ejemplo, con una muestra por día ($f_s = 1$ ciclo/día, $\xi_N = 0.5$), el ciclo diario ($\xi = 1$) se pliega sobre $|1 - 1\cdot1| = 0$: **la frecuencia cero, es decir, la media**. Dicho con la DFT: submuestrear una vez cada $m$ horas suma los coeficientes $\hat x[k]$ en pasos de $N/m$; en particular, la media de las mediciones de las 15 h es

$$\overline{x_{15}} = \frac1N\sum_{m=0}^{23}\hat x[365\,m]\,e^{2\pi i\, m\, 15/24},$$

que es la media general más el valor a las 15 h de todos los armónicos del ciclo diario. Es un sesgo, no ruido: cambia el promedio de la serie en una cantidad **sistemática** que depende de la hora elegida. Y cuando el período de muestreo no es un divisor del día, el ciclo diario aparece como una oscilación **falsa** de otra frecuencia: mirar una vez cada 25 horas ($f_s = 24/25 = 0.96$ ciclos/día) lo pliega sobre $|1 - 0.96| = 0.04$ ciclos/día, un ciclo de 25 días que en la serie original no existe (que exista un ciclo diario, sí).

**Qué mirar.** El espectro de la serie submuestreada (con `espectro.amplitudes(x, dt_nuevo)`, donde `dt_nuevo` es el nuevo paso en días) hasta su nueva frecuencia de Nyquist, y la comparación de medias.

### Tarea 6: Submuestrear a las 15 h y cada 25 h

1. Armá `T15`, la serie de 365 valores de las 15 h (`fecha.dt.hour == 15`), y calculá su espectro de amplitudes con `dt = 1` día. Graficalo junto con el espectro de amplitudes de la serie horaria (hasta 12 ciclos/día), con `semilogy`. ¿Dónde fue el ciclo diario? ¿Y el anual? Guardá `T15`, `media_15` y `media_horaria` y calculá la diferencia.
2. **Verificá la fórmula.** Guardá en `media_alias` el resultado de `np.real(sum(X[365*m] * np.exp(2j*np.pi*m*15/24) for m in range(24))) / N` con `X = np.fft.fft(T)`, y comprobá que es igual a `media_15`. Interpretalo: el sesgo es el valor a las 15 h del "ciclo diario medio". Calculá también la media de la serie submuestreada a las 3 h de la madrugada y a las 9 h. ¿Cuál es el signo del sesgo en cada caso?
3. **Aliasing genuino.** Submuestreá cada 25 horas: `T25 = T[::25]`, con paso `dt25 = 25/24` días. Predecí con la fórmula de arriba a qué frecuencia $\xi_{\text{alias}}$ (ciclos por día, `xi_alias_pred`) cae el ciclo diario, y verificalo: espectro de amplitudes de `T25` y amplitud máxima en el entorno $\xi_{\text{alias}} \pm 0.005$ comparada con la amplitud en ese mismo entorno de la serie horaria.

**Qué se espera.** El ciclo anual sigue en $\xi = 1/365$ con $A \approx 6.9$; el diario desapareció del espectro (con paso de 1 día solo se llega a $0.5$ ciclos/día) y su energía fue a la media: $\overline{x_{15}} \approx 21.5$ °C contra $18.2$ °C de la horaria. La fórmula da lo mismo hasta el redondeo. Con las 3 h el sesgo es negativo, con las 9 h también (la mañana es más fría que el promedio). Con la muestra cada 25 h aparece un ciclo de amplitud $\approx 2$ °C en $\xi \approx 0.04$ que en la serie horaria tiene menos de 1 °C (y de origen distinto: las fluctuaciones de los frentes): es un ciclo de 25 días que **no existe**.

In [ ]:
# TODO: T15 (365 valores), su espectro de amplitudes (dt = 1 día), y la figura
# TODO: media_15, media_horaria

X = np.fft.fft(T)
# TODO: media_alias = ... (fórmula con los X[365*m])
# TODO: medias a las 3 h y a las 9 h

# TODO: T25 = T[::25], dt25 = 25/24; xi_alias_pred; amplitud máxima cerca de xi_alias_pred (submuestreada y horaria)

In [ ]:
# Verificación
assert len(T15) == 365 and abs(media_15 - 21.54) < 0.05 and abs(media_horaria - 18.24) < 0.01, "medias: 21.5 (15 h) contra 18.2 (horaria)"
assert abs(espectro.amplitudes(T15, 1.0)[1][1] - 6.92) < 0.5, "el ciclo anual sigue ahí con la misma amplitud (dt = 1 día para T15)"
assert abs(media_alias - media_15) < 1e-6, "la fórmula de los coeficientes en múltiplos de 365 debe dar la media de las 15 h"
xi25_v, A25_v, _ = espectro.amplitudes(T[::25], 25 / 24)
assert abs(xi_alias_pred - 0.04) < 1e-9 and A25_v[abs(xi25_v - xi_alias_pred) < 0.005].max() > 1.5, "el ciclo diario reaparece como un ciclo de ~25 días"
print("submuestreo y aliasing: OK")

## 7. Filtrar: la parte regular, el residuo y los promedios móviles

### Filtrar en frecuencia

Un **filtro** en frecuencia multiplica el espectro por una función $H(\xi)$ y antitransforma: $y = \mathcal F^{-1}\bigl(H\,\hat x\bigr)$. Con $H \in \{0, 1\}$ se conservan algunas frecuencias y se descartan las otras. Para que $y$ sea real, $H$ tiene que ser simétrica ($H(-\xi) = H(\xi)$): con `rfft` eso es automático. `espectro.filtrar(x, dt, mascara)` lo hace: `mascara(xi)` recibe el arreglo de frecuencias (en ciclos por día) y devuelve 0/1 o números reales. Como los picos caen en frecuencias que son múltiplos de $1/365$, la máscara de picos es `np.isclose(xi, f, atol=medio_ancho)` con `medio_ancho` del orden de media resolución ($0.5/365$).

El filtro "sólo los picos" (más la media) da la **temperatura regular** $T_{\rm reg}$; lo que resta, $r = T - T_{\rm reg}$, es el **residuo**, la parte "irregular". Preguntas: ¿tiene estructura? Un ruido blanco (independiente de un instante al otro) tiene un espectro plano: $E[P[k]] = \sigma^2$ para todo $k$ (con la normalización $|\hat x[k]|^2/N$), y una autocorrelación nula fuera de $0$. Una señal cuyo espectro no es plano tiene *memoria*: el valor de hoy dice algo sobre el de mañana. Para medir el espectro de un residuo, que es una señal aleatoria, el periodograma es muy ruidoso (cada $P[k]$ tiene una dispersión del mismo orden que su valor, sin importar cuánto dure el registro); se lo suaviza promediando periodogramas de tramos (**método de Welch**, `espectro.welch(x, dt, tramo=...)`) a costa de resolución.

### Promedios móviles como filtros

El **promedio móvil** de largo $L$, $y_j = \frac1L\sum_{i=0}^{L-1}x_{j-i}$, es una **convolución** con $h = (1/L, \dots, 1/L)$, y por el teorema de convolución $\hat y[k] = \hat h[k]\,\hat x[k]$: **es un filtro**, con respuesta en frecuencia $H[k] = \hat h[k]$. Se calcula sumando una progresión geométrica: con $\xi_k = k/(N\Delta t)$,

$$|H(\xi)| = \left|\frac{\sin(\pi\xi L\,\Delta t)}{L\,\sin(\pi\xi\,\Delta t)}\right| \;\approx\; \bigl|\operatorname{sinc}(\xi\, L\Delta t)\bigr| \qquad (\xi\Delta t \ll 1),$$

con $\operatorname{sinc}(u) = \sin(\pi u)/(\pi u)$. Las consecuencias:

* $H(0) = 1$: conserva la media y las variaciones más lentas que la ventana ($\xi \ll 1/(L\Delta t)$).
* **Ceros en los múltiplos de $1/(L\Delta t)$**: el promedio de *cualquier* señal periódica de período $L\Delta t/m$ sobre una ventana de $L\Delta t$ es exactamente cero. Por eso un promedio móvil de 24 h (una ventana de 1 día, ceros en $1, 2, 3, \dots$ ciclos/día) **borra exactamente el ciclo diario y todos sus armónicos**, y uno de 7 días ($L = 168$, ceros en $1/7, 2/7, \dots$ ciclos/día, que incluyen $1, 2, 3$) los borra también, y además atenúa todo lo que tenga períodos más cortos que una semana.
* **Lóbulos laterales**: entre los ceros $|H|$ no es cero, sube hasta unos $1/(\pi\,\xi L\Delta t)$: el promedio de 24 h deja pasar alrededor del $20\ \%$ de una oscilación de 16 h ($\xi = 1.5$). Es un filtro pasabajos *malo*: no corta, atenúa a los saltos, y cambia de signo ($H<0$ en los lóbulos impares: la señal filtrada tiene invertidas algunas frecuencias).

Contra el filtro "ideal" de picos (que conserva exactamente lo que se elige y anula el resto), el promedio móvil tiene la ventaja de que **no hay que elegir las frecuencias** (solo un ancho) y de ser causal y barato; pero no es selectivo. Y hacen *cosas opuestas*: el filtro de picos **se queda** con los ciclos y tira lo irregular, el promedio de 24 h **tira** el ciclo diario y se queda con lo irregular lento y el anual. Para compararlos tenés que mirarlos con ese cuidado.

Para calcular un promedio móvil centrado y periódico (la DFT supone que la serie se repite), usá `scipy.ndimage.uniform_filter1d(x, size=L, mode="wrap")`: así $|\hat y[k]/\hat x[k]| = |H(\xi_k)|$ es **exacto** y se puede comparar con la fórmula.

### Tarea 7: Temperatura regular, residuo y promedios móviles

1. **Filtro de picos.** Con `espectro.filtrar`, construí `T_reg` que conserva la media y los picos anual, semianual y diario con sus armónicos de 12 h y 8 h ($k = 1, 2, 365, 730, 1095$). Calculá `resid = T - T_reg`. Graficá un mes (por ejemplo mayo) con la señal, la temperatura regular y el residuo. Chequeá que la fracción de la energía del residuo es $1 - $ `frac_picos` (Tarea 4).
2. **¿Es blanco el residuo?** Calculá su periodograma y su espectro de Welch (`espectro.welch(resid, dt, tramo=45*24)`). Graficalos en log-log junto con la línea del ruido blanco de la misma varianza ($P = \sigma^2$, `resid.var()`), y marcá los períodos de 1 día, 1 semana y 1 mes. Calculá la fracción de la energía del residuo en las bandas de período: más de 30 días, entre 7 y 30, entre 2 y 7, entre 1 y 2, y menos de 1 día (`fracs_banda`, un diccionario o lista con estas cinco fracciones, que suman 1). ¿En qué escalas tiene más energía? Calculá la autocorrelación a 1 hora y a 24 horas.
3. **Promedios móviles.** Con `uniform_filter1d(..., mode="wrap")` calculá `T_ma24` y `T_ma7d` ($L = 24$ y $L = 168$). Graficalos sobre un mes junto con la señal. Después, estimá la **respuesta en frecuencia empírica** de cada uno, `H_emp = |rfft(y)| / |rfft(T)|` (en los coeficientes $k$ donde `|rfft(T)|` no es despreciable), y compará con la fórmula teórica, que escribís como una función `H_teo(xi, L)` (¡cuidado con $\xi = 0$!). Graficá `|H|` contra $\xi$ para $\xi \in [0, 3]$ ciclos/día (curva teórica continua y puntos empíricos), marcando los ceros. Verificá que `|H_emp|` es cero (a $10^{-10}$ o menos) en $\xi = 1, 2$ para los dos promedios. Los ceros del de 7 días en $\xi = 1/7, 2/7, \dots$ no se ven en la DFT de un año ($365/7$ no es entero: esas frecuencias no están en la grilla); verificalos con la fórmula teórica.
4. Respondé: ¿qué elimina cada promedio y qué deja? ¿En qué se diferencia el promedio de 24 h de "T menos el ciclo diario" (`T - ` la parte diaria del filtro de picos)? Miralo con las series y con los espectros.

**Qué se espera.** El residuo tiene un desvío de unos $3.4$ °C (29 % de la varianza) y un espectro que **no es plano**: $P$ baja unos cuatro órdenes de magnitud entre las escalas de semanas y de horas, con una pendiente de log-log cercana a $-2$, y la mayor parte de la energía en escalas de 2 a 30 días (los frentes, las masas de aire). La autocorrelación a 1 h es cercana a 1, así que el residuo es cualquier cosa menos independiente de un instante al otro. El promedio de 24 h anula el ciclo diario (`|H|` del orden de $10^{-16}$ en $\xi = 1$ y 2) pero deja pasar bastante en $\xi \approx 1.5$ y $2.5$; el de 7 días también, y atenúa mucho más las fluctuaciones de menos de una semana.

In [ ]:
from scipy.ndimage import uniform_filter1d

picos = [1, 2, 365, 730, 1095]
def mascara(xi):
    # 1 en la media y en los picos, 0 en el resto
    ...  # TODO

# TODO: T_reg = espectro.filtrar(T, dt, mascara); resid = T - T_reg; figura de un mes
# TODO: periodograma y welch del residuo; línea de ruido blanco; fracs_banda (5 bandas); autocorrelaciones

# TODO: T_ma24 y T_ma7d con uniform_filter1d(..., mode="wrap")
# TODO: H_emp de cada uno, fórmula teórica H_teo(xi, L), figura y verificación de los ceros

In [ ]:
# Verificación
assert np.isclose(resid.var() / T.var(), 1 - frac_picos, atol=2e-3), "la energía del residuo debe ser 1 - la fracción de los picos"
assert abs(sum(fracs_banda) - 1) < 1e-6 and max(fracs_banda) < 0.6, "las cinco bandas suman 1; el residuo tiene energía repartida en varias escalas"
assert abs(np.corrcoef(resid[:-1], resid[1:])[0, 1]) > 0.5, "el residuo no es blanco: la autocorrelación a 1 h es alta"
xi_v = np.fft.rfftfreq(N, dt)
for y in (T_ma24, T_ma7d):
    H = np.abs(np.fft.rfft(y)) / np.abs(np.fft.rfft(T))
    assert H[365] < 1e-8 and H[730] < 1e-8, "ambos promedios anulan el ciclo diario y su armónico de 12 h"
    assert np.abs(H[xi_v > 0] - H_teo(xi_v[xi_v > 0], 24 if y is T_ma24 else 168))[np.abs(np.fft.rfft(T))[xi_v > 0] > 1].max() < 1e-6, "la respuesta empírica debe coincidir con la fórmula"
assert H_teo(1 / 7, 168) < 1e-8 and abs(H_teo(0.5, 24) - 0.6366) < 1e-3, "H_teo: ceros en múltiplos de 1/(L dt)"
print("filtrado y promedios móviles: OK")

## 8. Compresión por coeficientes de mayor módulo

### Idea

Si la señal se puede describir bien con pocos coeficientes de Fourier, se guardan solo esos y se descarta el resto. La versión más simple es **conservar los $M$ coeficientes $\hat x[k]$ de mayor módulo** (con sus índices) y antitransformar $x_M = \mathcal F^{-1}(\hat x_M)$, con $\hat x_M[k] = \hat x[k]$ si $k$ está entre los $M$ mayores y $0$ si no. Como la señal es real, $|\hat x[N-k]| = |\hat x[k]|$ y el conjunto de los $M$ mayores viene en pares $\pm k$ (salvo que el corte de $M$ separe un par: en ese caso $x_M$ es levemente compleja; tomá la parte real).

### El error se calcula sin reconstruir

Por Plancherel, $\|x - x_M\|_2^2 = \frac1N\sum_{k\notin \text{conservados}}|\hat x[k]|^2$: el error es la energía de los coeficientes descartados. Si se ordenan los $|\hat x[k]|^2$ de mayor a menor, $E_{(1)} \ge E_{(2)} \ge \dots$, el **error relativo** es

$$e(M) = \frac{\|x - x_M\|_2}{\|x\|_2} = \sqrt{\frac{\sum_{i > M}E_{(i)}}{\sum_{i}E_{(i)}}},$$

una suma acumulada al revés: no hace falta ninguna antitransformada. Se la verifica con una reconstrucción explícita para unos pocos valores de $M$.

### Cuidado con la norma

El error relativo respecto de $\|x\|_2$ incluye la media: si la media es grande respecto de las fluctuaciones (acá 18 °C contra 6 °C), un $M = 1$ (solo la media) ya da un error moderado, y la cifra "5 %" es más generosa de lo que parece. Es más honesto mirarlo también respecto de $\|x - \bar x\|_2$, que es el error relativo *de las fluctuaciones*, y decir cuál se usa.

### Comparación con la imagen

En la imagen del Capítulo 15 (`datos/astronauta.png`, $512\times512$) la compresión funciona con la DFT bidimensional (`np.fft.fft2`) de la misma manera: ordenar los $512^2$ coeficientes por módulo y conservar los mayores. Los dos casos se comparan en una gráfica **error contra fracción conservada** ($M/N$, con $N$ el número de coeficientes), en escala log-log, porque las señales tienen tamaños distintos.

### Tarea 8: Compresión de la serie y comparación con la imagen

1. Calculá `X = np.fft.fft(T)`, ordená los coeficientes por módulo decreciente (`orden = np.argsort(-np.abs(X))`) y calculá `err_rel[M-1]` = error relativo con los $M$ mayores, para $M = 1, \dots, N-1$, con la fórmula de Plancherel (suma acumulada), respecto de $\|T\|_2$; y `err_fluct` respecto de $\|T - \bar T\|_2$.
2. Verificá la fórmula reconstruyendo realmente con $M = 11$, $101$ y $1001$: función `comprimir(M)` que devuelve `np.fft.ifft(Xm).real`.
3. Graficá el error relativo contra la fracción conservada $M/N$ (log-log) y marcá las fracciones del 1 %, el 0.1 % y el 0.01 % ($M = 88, 9, 1$ coeficientes) con sus errores. ¿Cuántos coeficientes hacen falta para un error del 5 % respecto de $\|T\|$? Guardá `M_5`. ¿Y respecto de $\|T - \bar T\|$ (`M_5_fluct`)? ¿Qué son (qué frecuencias) los primeros 20 coeficientes conservados?
4. Graficá en dos semanas de enero la señal original, la reconstruida con `M_5` coeficientes y con el 1 %, y el error.
5. Repetí el análisis de (1) para la imagen: `img = plt.imread(datos.obtener("astronauta.png"))` (tomá un solo canal si es de 3), `np.fft.fft2`, y superponé su curva error contra fracción en el gráfico de (3). ¿Cuál se comprime mejor? ¿Hay alguna diferencia de fondo entre las dos señales que lo explique?

**Qué se espera.** Con el 1 % de los coeficientes (88) el error de la serie es del orden del $10\%$; con $0.1\%$ (9), del $17\%$; con $0.01\%$ (un solo coeficiente: la media), del $33\%$. Para un $5\%$ hacen falta unos $460$ coeficientes ($\approx 5\%$ de $N$), y muchos más ($\approx 2000$, un cuarto de $N$) para el $5\%$ de las fluctuaciones: la parte irregular es cara de describir con Fourier. Respecto de la norma total, la serie se comprime *mejor* que la imagen (con el 1 %, $11\%$ contra $15\%$; el 5 % de error cuesta el $5\%$ de los coeficientes en la serie y el $17\%$ en la imagen), pero es un efecto de la media (la serie tiene una media enorme respecto de sus fluctuaciones); respecto de las fluctuaciones ocurre al revés y por poco (con el 1 %: $33\%$ la serie, $27\%$ la imagen). Para tus conclusiones, mirá cuántos números hay en los picos (11) y cuánto error tiene la serie con solo ellos, y qué son las frecuencias de los primeros coeficientes que siguen.

In [ ]:
X = np.fft.fft(T)
orden = np.argsort(-np.abs(X))
# TODO: E_ord = |X[orden]|^2 / N; err_rel[M-1] y err_fluct[M-1] con sumas acumuladas (Plancherel)

def comprimir(M):
    # antitransformada de los M coeficientes de mayor módulo (parte real)
    ...  # TODO

# TODO: verificar con M = 11, 101, 1001 reconstruyendo
# TODO: M_5 y M_5_fluct; los primeros 20 coeficientes conservados: ¿qué frecuencias?
# TODO: gráfico error contra M/N (log-log), con la imagen superpuesta; gráfico de dos semanas de enero

In [ ]:
# Verificación
assert abs(err_rel[100] - np.linalg.norm(T - comprimir(101)) / np.linalg.norm(T)) < 1e-6, "Plancherel debe coincidir con la reconstrucción"
assert 440 <= M_5 <= 480, "para el 5 % respecto de ||T|| hacen falta ~460 coeficientes"
assert M_5_fluct > 3 * M_5, "respecto de las fluctuaciones hacen falta muchos más"
assert abs(err_rel[87] - 0.107) < 0.01 and abs(err_rel[0] - 0.327) < 0.01, "1 % (M = 88): ~0.107; M = 1: ~0.327"
print("compresión: OK")

## 9. Espectrograma: cuando el contenido de frecuencias cambia

Un único espectro promedia sobre todo el registro: si la amplitud del ciclo diario cambia con la estación, o si una nota musical decae mientras suena, esa información está en el espectro pero *mezclada* (está en las fases de los coeficientes de la banda). El **espectrograma** o transformada de Fourier de tiempo corto (STFT) la muestra: se parte la señal en tramos de $N_v$ muestras (ventanas), separados por un **paso** de $p$ muestras, se calcula el periodograma de cada tramo (con una ventana de Hann para controlar la fuga) y se grafica la matriz $S[k, j]$ (potencia en la frecuencia $\xi_k$ del tramo $j$) como una imagen: tiempo en el eje horizontal (el del centro del tramo), frecuencia en el vertical y potencia en color, casi siempre en **escala logarítmica** (decibeles, $10\log_{10}S$).

### Los parámetros y el compromiso tiempo–frecuencia

* **Ventana** de duración $\tau = N_v\Delta t$: da la **resolución en frecuencia** $\Delta\xi = 1/\tau$ (Sección 1: es $1/T$ del tramo) y la **resolución temporal**: un cambio que dure menos que $\tau$ no se resuelve.
* Vale siempre $\Delta t_{\rm res}\cdot\Delta\xi \gtrsim 1$: **no se puede tener las dos resoluciones a la vez** (es el principio de incertidumbre de Fourier: una señal corta en el tiempo tiene un espectro ancho). Ventanas cortas, buena resolución temporal y espectro borroso; ventanas largas, picos finos pero las variaciones rápidas se promedian. No hay una ventana "correcta": depende de qué se quiere ver, y por eso es útil probar varias.
* **Solapamiento**: pasos $p < N_v$ (típico $p = N_v/2$, solapamiento del 50 %) evitan que un evento caiga en el borde de las ventanas, donde Hann lo atenúa. Solapar más da una imagen más suave pero no información nueva.

### Amplitud a partir de un espectrograma

Con Hann, un coseno de amplitud $A$ en la grilla da $|\hat{(xw)}[k]| = A\sum w/2$ y, con la normalización de `imc.espectro`, $S = |\hat{(xw)}|^2/\sum w^2$. Entonces $A = 2\sqrt{S\sum w^2}/\sum w$. Eso permite leer amplitudes en °C de la imagen, y compararlas con las que da un ajuste por mínimos cuadrados en cada ventana (el estimador "fuerte" de la Sección 3).

### La guitarra

`cuerda_guitarra.wav` (ver `datos/README.md`) es una cuerda de guitarra pulsada, mono a 44.1 kHz, 6 s. El **audio** cambia con el tiempo: una nota musical no es una frecuencia sino una **fundamental** más **armónicos** (múltiplos enteros: $2f_0, 3f_0, \dots$) cuyas amplitudes relativas definen el timbre, y que **decaen a distintas velocidades**. El enunciado de las notas habla de "una escala tocada en un instrumento y grabada con el teléfono", para identificar las notas; acá usamos la cuerda del repositorio, que no cambia de nota pero sí de amplitud y de contenido: es una buena razón para el espectrograma. El nombre de la nota sale de la frecuencia con la escala temperada: la nota MIDI es $n = 69 + 12\log_2(f/440)$ (La4 = 440 Hz = 69; los enteros son notas; cada 12 son una octava).

### Tarea 9: El espectrograma: temperatura y guitarra

1. **Implementación simple.** Escribí `espectrograma_simple(x, dt, tramo, paso)` que devuelve `(t_centros, xi, S)` con `S[k, j]` = periodograma del tramo $j$ con ventana de Hann, restando la media del tramo y con la normalización $|\hat{(xw)}[k]|^2/(N_v\,\overline{w^2})$ (la de `espectro.periodograma`). Comprobá que coincide con `espectro.espectrograma(x, dt, tramo, solapamiento=1 - paso/tramo)` para la temperatura (guardá el resultado propio como `ts, xi_s, S`).
2. **Temperatura.** Usá la serie de los tres años (`df.temp.values`) con ventanas de 30 días (720 horas) y paso de 15 días (360). Graficá el espectrograma con `pcolormesh` (fechas en el eje horizontal, $\xi \in [0, 4]$ ciclos/día en el vertical, potencia en $\log_{10}$). Calculá, para cada ventana, la amplitud del ciclo diario de dos maneras: (a) la que se lee de la fila $\xi = 1$ del espectrograma con $A = 2\sqrt{S\sum w^2}/\sum w$, y (b) por mínimos cuadrados con $\xi = 1$ fijo (`ajuste_ls` de la Tarea 3) en el tramo. Guardalas en `A_esp` y `A_min` y graficá las dos curvas contra la fecha. ¿En qué época del año es máxima la amplitud del ciclo diario y en cuál mínima? ¿Es el mismo patrón los tres años?
3. **Guitarra.** Cargá `datos.obtener("cuerda_guitarra.wav")` con `wavfile.read`, pasala a `float` y normalizala por 32768. Calculá el espectrograma con ventanas de 5 ms, 50 ms y 500 ms (paso = mitad de la ventana) y graficá los tres en dB (`10*np.log10(S + 1e-12)`, con un rango dinámico de unos 60–70 dB) para $\xi \in [0, 2000]$ Hz, con las ventanas de 5 ms, 50 ms y 500 ms en paneles vecinos.
4. **La nota y sus armónicos.** Con el periodograma con Hann de los primeros 3 s (`espectro.periodograma`), encontrá la frecuencia fundamental `f0` en Hz (el máximo en $[100, 400]$ Hz), decí de qué nota se trata (MIDI y nombre) y qué tan desafinada está (en *cents*: $1200\log_2(f_0/f_{\rm nota})$). Después, con el espectrograma de 50 ms, seguí la evolución temporal de la fundamental y de sus primeros cuatro armónicos (la potencia máxima en $\pm 20$ Hz de $m f_0$) y estimá para cada uno su velocidad de decaimiento en dB/s entre $t = 0.2$ y $2.5$ s (recta de mínimos cuadrados en dB). Guardá `decaimiento` (arreglo de 5, en dB/s, negativos).
5. **Compromiso.** Con los tres espectrogramas de (3): ¿qué se ve y qué no se ve con 5 ms? ¿Y con 500 ms? ¿Cuánto vale $\Delta\xi$ para cada ventana y cómo se compara con la separación entre armónicos ($f_0 \approx 196$ Hz)? Lo mismo con la temperatura: ¿qué pasaría con una ventana de 3 días? ¿Y de 1 año?

**Qué se espera.** En la temperatura: una raya horizontal brillante en $\xi = 1$ (y otra más débil en $\xi = 2$) cuyo brillo sube y baja con las estaciones; las amplitudes de (a) y (b) siguen el mismo patrón general pero no coinciden (diferencias de hasta medio grado en algunas ventanas): son estimadores distintos, porque Hann pondera el centro de la ventana de 30 días y descarta casi por completo los bordes, mientras que mínimos cuadrados usa todo el tramo con el mismo peso. En la guitarra: una fundamental en $\approx 196$ Hz (Sol$_3$) con armónicos en $\approx 392, 588, 784, \dots$ Hz que decaen a velocidades distintas (los agudos más rápido); con 5 ms ($\Delta\xi = 200$ Hz) los armónicos se funden en una sola mancha; con 500 ms ($\Delta\xi = 2$ Hz) cada armónico es una raya finita pero el ataque (el pulsado) se "difumina" en el tiempo; con 50 ms ($\Delta\xi = 20$ Hz) se ve el mejor compromiso.

In [ ]:
def espectrograma_simple(x, dt, tramo, paso):
    # devuelve (t_centros, xi, S) con S[k, j] = periodograma (Hann) del tramo j
    x = np.asarray(x, float)
    w = np.hanning(tramo)
    xi = np.fft.rfftfreq(tramo, d=dt)
    ts, cols = [], []
    for i in range(0, len(x) - tramo + 1, paso):
        seg = x[i:i + tramo]
        # TODO: restar la media del tramo, multiplicar por w, rfft, |.|^2 / (tramo * mean(w^2))
        ts.append((i + tramo / 2) * dt)
    return np.array(ts), xi, np.array(cols).T

# TODO: comprobar contra espectro.espectrograma en la temperatura
# TODO: temperatura de los 3 años, ventana de 30 días: figura, amplitud diaria (a) del espectrograma y (b) por mínimos cuadrados
# TODO: guitarra: lectura del wav y tres espectrogramas (5 ms, 50 ms, 500 ms)
# TODO: f0, nota, desafinación (cents); seguimiento de los armónicos; decaimiento (dB/s)

In [ ]:
# Verificación
_, _, S_imc = espectro.espectrograma(df.temp.values, dt, 720, solapamiento=0.5)
assert np.allclose(S, S_imc, rtol=1e-8, atol=1e-10), "tu espectrograma debe coincidir con el de imc (resta de la media, Hann de np.hanning, |X|^2 / (N mean(w^2)))"
assert np.corrcoef(A_esp, A_min)[0, 1] > 0.75, "las dos amplitudes diarias deben tener el mismo patrón general"
assert 194 < f0 < 198 and round(69 + 12 * np.log2(f0 / 440)) == 55, "la fundamental es ~196 Hz: Sol3 (MIDI 55)"
assert np.all(decaimiento < -5) and decaimiento[3] < decaimiento[0], "todos decaen, los agudos más rápido que la fundamental"
print("espectrogramas: OK")

## Interpretación

Esta es la parte que se evalúa. Respondé en las celdas de texto que siguen, con oraciones completas y citando los números o gráficos que obtuviste.

**1.** **Frecuencias, amplitudes y fases.** ¿Qué frecuencias contiene la serie de temperatura? Da una tabla o una lista con las amplitudes (°C), las fechas u horas de los máximos y la fracción de la varianza que explica cada componente (Tareas 2 a 4). ¿Cuánto de la variabilidad total explican los ciclos diario y anual, con sus armónicos, y qué significa ese número? Explicá por qué la amplitud del ciclo diario no es una constante sino que cambia con la estación (Tareas 3 y 9) y por qué al estimarla en un tramo corto la DFT y el ajuste por mínimos cuadrados pueden dar valores distintos y cuál conviene.

*(escribí tu respuesta acá)*

**2.** **La parte regular y lo que queda.** Describí cómo separaste la temperatura regular del residuo (Tarea 7) y qué queda: ¿es ruido blanco? Fundamentá con el espectro, la autocorrelación y las escalas de tiempo de mayor energía; asociá esas escalas con algo físico. Comparalo con el promedio móvil de 24 h y con el de 7 días: explicá con la respuesta en frecuencia (ceros, lóbulos) qué borra exactamente cada uno y en qué es peor que el filtro de picos. Además: ¿qué efecto tiene la fuga espectral y la ventana de Hann sobre la lectura del espectro entre los picos (Tarea 5)?

*(escribí tu respuesta acá)*

**3.** **Muestrear menos.** Si en lugar de una medición por hora hubiera una por día, o una por semana: ¿qué información se pierde y cuál se falsea? Explicalo con tus resultados de la Tarea 6: a dónde fue el ciclo diario, cuánto cambió la media, por qué depende de la hora elegida, y qué pasa cuando el intervalo de muestreo no divide al día (los 25 h). Formulá una regla práctica (con la frecuencia de Nyquist) para decidir cada cuánto hay que medir para no falsear el ciclo diario.

*(escribí tu respuesta acá)*

**4.** **Cuántos números y qué se pierde.** ¿Cuántos coeficientes hacen falta para reconstruir la serie con un error del 5 %? Distinguí el error respecto de la norma de la señal y respecto de las fluctuaciones (Tarea 8) y decí cuál es más honesto. Comparalo con la imagen. Escribí el párrafo de conclusión que pide el ejercicio de las notas: qué información contiene esta serie, en cuántos números se la puede resumir y qué se pierde al hacerlo.

*(escribí tu respuesta acá)*

**5.** **¿Sirven las mismas herramientas para otras señales?** Con lo que hiciste con la guitarra (Tarea 9), la imagen (Tarea 8) y la temperatura: ¿qué herramienta sirvió igual para las tres y cuál no? ¿Qué cambia cuando la señal deja de ser estacionaria (el contenido de frecuencias cambia con el tiempo) y cómo lo resuelve el espectrograma, a qué costo? ¿Qué elegirías para un electrocardiograma (una señal casi periódica con un latido de forma fija, que cambia de ritmo) y por qué?

*(escribí tu respuesta acá)*